## Flow_matching: PushT-transformers

---- 

- conda env : [flow_matching](./README.md#setup-a-conda-environment)

---
### Reference:

- ***Github***:
    - https://github.com/HRI-EU/flow_matching

In [1]:
import sys
sys.path.append('./temp_flow_matching/external')
import numpy as np

import torch
import temp_flow_matching.external.models.pusht as pusht
import torch.nn as nn
from tqdm import tqdm
from models.TransformerForDiffusion import TransformerForDiffusion
from models.resnet import get_resnet
from models.resnet import replace_bn_with_gn
import collections
from diffusers.training_utils import EMAModel
from torch.utils.data import Dataset, DataLoader
from diffusers.optimization import get_scheduler
from torchcfm.conditional_flow_matching import *
from torchcfm.utils import *
from torchcfm.models.models import *
from termcolor import colored
from skvideo.io import vwrite
from IPython.display import Video
import cv2

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/home/hyunjae/.local/lib/python3.10/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

### Download the pusht data and put in the folder

In [3]:
import os
from pathlib import Path
import gdown
import zipfile

Path("./temp_data").mkdir(exist_ok=True, parents=True)
zipfile_path = "./temp_data/pusht_cchi_v7_replay.zarr.zip"
if not os.path.isfile(zipfile_path):
    id = "1KY1InLurpMvJDRb14L9NlXT_fEsCvVUq&confirm=t"
    gdown.download(id=id, output=zipfile_path, quiet=False)

dataset_path = "./temp_data/pusht_cchi_v7_replay.zarr"
with zipfile.ZipFile(zipfile_path, 'r') as zip_ref:
    zip_ref.extractall(dataset_path)

In [4]:
obs_horizon = 1
pred_horizon = 16
action_dim = 2
action_horizon = 8
num_epochs = 3001
vision_feature_dim = 514
batch_size = 256

In [5]:
# create dataset from file
dataset = pusht.PushTImageDataset(
    dataset_path=dataset_path,
    pred_horizon=pred_horizon,
    obs_horizon=obs_horizon,
    action_horizon=action_horizon
)
# save training data statistics (min, max) for each dim
stats = dataset.stats

# create dataloader
dataloader = DataLoader(
    dataset,
    batch_size= batch_size, #64,
    num_workers=1,
    shuffle=True,
    # accelerate cpu-gpu transfer
    pin_memory=True,
    # don't kill worker process afte each epoch
    persistent_workers=True
)

### Create Network

In [6]:
vision_encoder = get_resnet('resnet18')
vision_encoder = replace_bn_with_gn(vision_encoder)
noise_pred_net = TransformerForDiffusion(
    input_dim=action_dim,
    output_dim=action_dim,
    horizon=pred_horizon,
    cond_dim=vision_feature_dim
)
nets = nn.ModuleDict({
    'vision_encoder': vision_encoder,
    'noise_pred_net': noise_pred_net
}).to(device)

In [7]:
sigma = 0.0
ema = EMAModel(
    parameters=nets.parameters(),
    power=0.75)
optimizer = torch.optim.AdamW(params=nets.parameters(), lr=1e-4, weight_decay=1e-6)
lr_scheduler = get_scheduler(
    name='cosine',
    optimizer=optimizer,
    num_warmup_steps=500,
    num_training_steps=len(dataloader) * num_epochs
)

FM = ConditionalFlowMatcher(sigma=sigma)

In [8]:
def test(ema_nets):
    # # PATH = './checkpoint_t/flow_ema_03000.pth'
    # PATH = './checkpoint_t/flow_ema_00020.pth'
    # state_dict = torch.load(PATH, map_location='cuda')
    # ema_nets = nets
    # ema_nets.vision_encoder.load_state_dict(state_dict['vision_encoder'])
    # ema_nets.noise_pred_net.load_state_dict(state_dict['noise_pred_net'])

    max_steps = 300
    env = pusht.PushTImageEnv()
    env.seed(100000)

    # get first observation
    obs, info = env.reset()

    # keep a queue of last 2 steps of observations
    obs_deque = collections.deque(
        [obs] * obs_horizon, maxlen=obs_horizon)
    # save visualization and rewards
    imgs = [env.render(mode='rgb_array')]
    rewards = list()
    done = False
    step_idx = 0
    
    with tqdm(total=max_steps, desc="Eval PushTImageEnv") as pbar:
        while not done:
            B = 1
            x_img = np.stack([x['image'] for x in obs_deque])
            x_pos = np.stack([x['agent_pos'] for x in obs_deque])
            x_pos = pusht.normalize_data(x_pos, stats=stats['agent_pos'])

            x_img = torch.from_numpy(x_img).to(device, dtype=torch.float32)
            x_pos = torch.from_numpy(x_pos).to(device, dtype=torch.float32)
            # infer action
            with torch.no_grad():
                # get image features
                # t1 = time.time()
                # image_features = ema_nets['vision_encoder'](x_img)
                # obs_features = torch.cat([image_features, x_pos], dim=-1)
                # obs_cond = obs_features.unsqueeze(0).flatten(start_dim=1)

                # image_features = ema_nets['vision_encoder'](x_img.flatten(end_dim=1))
                image_features = ema_nets['vision_encoder'](x_img)
                # image_features = image_features.reshape(*x_img.shape[:2], -1)
                image_features = image_features.unsqueeze(1)
                obs_features = torch.cat([image_features, x_pos.unsqueeze(1)], dim=-1)
                # obs_cond = obs_features.flatten(start_dim=1)
                obs_cond = obs_features

                timehorion = 1
                for i in range(timehorion):
                    noise = torch.rand(1, pred_horizon, action_dim).to(device)
                    x0 = noise.expand(x_img.shape[0], -1, -1)
                    timestep = torch.tensor([i / timehorion]).to(device)

                    if i == 0:
                        vt = nets['noise_pred_net'](x0, timestep, obs_cond)
                        traj = (vt * 1 / timehorion + x0)

                    else:
                        vt = nets['noise_pred_net'](traj, timestep, obs_cond)
                        traj = (vt * 1 / timehorion + traj)

            # print(time.time() - t1)

            naction = traj.detach().to('cpu').numpy()
            naction = naction[0]
            action_pred = pusht.unnormalize_data(naction, stats=stats['action'])

            # only take action_horizon number of actions
            start = obs_horizon - 1
            end = start + action_horizon
            action = action_pred[start:end, :]

            # x_img = x_img[0, :].permute((1, 2, 0))
            # plot_trajectory(x0[0].detach().cpu().numpy(), vt[0].detach().cpu().numpy(),
            #                 action_pred,
            #                 x_img.detach().cpu().numpy())

            # execute action_horizon number of steps
            for j in range(len(action)):
                # stepping env
                obs, reward, done, _, info = env.step(action[j])
                # save observations
                obs_deque.append(obs)
                # and reward/vis
                rewards.append(reward)
                imgs.append(env.render(mode='rgb_array'))

                # update progress bar
                step_idx += 1

                pbar.update(1)
                pbar.set_postfix(reward=reward)

                if step_idx > max_steps:
                    done = True
                if done:
                    break

    return imgs, rewards

In [9]:
def train(output_dir, save_freq = 1000, eval_freq = 1000):
    # imgs, rewards = test(nets)
    
    Path(output_dir).mkdir(exist_ok=True, parents=True)
    ckp_model_dir = os.path.join(output_dir, "checkpoint_t")
    Path(ckp_model_dir).mkdir(exist_ok=True, parents=True)
    avg_loss_train_list = []
    tqdm_epochs = tqdm(range(num_epochs))
    for epoch in tqdm_epochs:
        total_loss_train = 0.0
        for data in dataloader:
            x_img = data['image'][:, :obs_horizon].to(device)
            x_pos = data['agent_pos'][:, :obs_horizon].to(device)
            x_traj = data['action'].to(device)

            x_traj = x_traj.float()
            x0 = torch.randn(x_traj.shape, device=device)
            timestep, xt, ut = FM.sample_location_and_conditional_flow(x0, x_traj)

            # encoder vision features
            image_features = nets['vision_encoder'](x_img.flatten(end_dim=1))
            image_features = image_features.reshape(*x_img.shape[:2], -1)
            obs_features = torch.cat([image_features, x_pos], dim=-1)
            # obs_cond = obs_features.flatten(start_dim=1)
            obs_cond = obs_features

            vt = nets['noise_pred_net'](xt, timestep, obs_cond)

            loss = torch.mean((vt - ut) ** 2)
            total_loss_train += loss.detach()

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            lr_scheduler.step()

            # update Exponential Moving Average of the model weights
            ema.step(nets.parameters())

        avg_loss_train = total_loss_train / len(dataloader)
        avg_loss_train_list.append(avg_loss_train.detach().cpu().numpy())
        # print(colored(f"epoch: {epoch:>02},  loss_train: {avg_loss_train:.10f}", 'yellow'))
        # tqdm_epochs.write(f"epoch: {epoch:>02},  loss_train: {avg_loss_train:.10f}")
        tqdm_epochs.set_description(f"epoch: {epoch:>02},  loss_train: {avg_loss_train:.10f}")

        if epoch % eval_freq == 0 or epoch == num_epochs - 1:
            imgs, rewards = test(nets)
            print('Evaluation {epoch:05d} Score: ', max(rewards))
            vid_output_path = os.path.join(output_dir, f"eval_{epoch:05d}.mp4")
            size = (imgs[0].shape[1],imgs[0].shape[0])
            out = cv2.VideoWriter(vid_output_path,cv2.VideoWriter_fourcc(*'mp4v'),15, size)
            for i in range(len(imgs)):
                out.write(imgs[i])
            out.release()
        
        if epoch % save_freq == 0 or epoch == num_epochs - 1:
            ema.store(nets.parameters())
            ema.copy_to(nets.parameters())
            # PATH = './checkpoint_t/flow_ema_%05d.pth' % epoch
            output_mode_path = os.path.join(ckp_model_dir, f"flow_ema_{epoch:05d}.pth")
            torch.save({'vision_encoder': nets.vision_encoder.state_dict(),
                        'noise_pred_net': nets.noise_pred_net.state_dict(),
                        }, output_mode_path)
            ema.restore(nets.parameters())


In [10]:
train("temp_output/pushT_transformer", save_freq = 250, eval_freq = 250)

Eval PushTImageEnv: 301it [00:03, 96.48it/s, reward=0] 0/3001 [00:43<?, ?it/s]


Evaluation {epoch:05d} Score:  0.0


Eval PushTImageEnv: 301it [00:03, 87.21it/s, reward=0]  250/3001 [3:00:32<32:48:30, 42.93s/it]


Evaluation {epoch:05d} Score:  0.2702971402296157


Eval PushTImageEnv: 301it [00:02, 107.92it/s, reward=0.81]0/3001 [6:00:14<29:58:55, 43.16s/it]


...

Eval PushTImageEnv:  39%|███▉      | 117/300 [00:01<00:01, 109.62it/s, reward=1]0:32, 43.16s/it]


Evaluation {epoch:05d} Score:  1.0


Eval PushTImageEnv: 301it [00:02, 120.59it/s, reward=0.304]00/3001 [36:00:20<00:43, 43.31s/it]  


Evaluation {epoch:05d} Score:  0.6982671104151519


epoch: 3000,  loss_train: 0.0162075236: 100%|██████████| 3001/3001 [36:00:23<00:00, 43.19s/it]
